In [ ]:
import os
import shutil
from dotenv import load_dotenv, find_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_classic.chains.question_answering import load_qa_chain


In [ ]:
load_dotenv(find_dotenv(usecwd=True), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY não encontrada no .env"


In [ ]:
embeddings_models = OpenAIEmbeddings()
llm = ChatOpenAI(model="gpt-4o-mini", max_tokens=2000)


In [ ]:
PDF_PATH = "../os-sertoes.pdf"
PERSIST_DIR = "text_index"
REBUILD = True  # deixe True só na primeira execução; depois mude para False

def limpar(texto):
    linhas = [
        l for l in texto.splitlines()
        if "b i b l i" not in l and ".indd" not in l
    ]
    return "\n".join(linhas).strip()

if REBUILD or not os.path.exists(PERSIST_DIR):
    shutil.rmtree(PERSIST_DIR, ignore_errors=True)

    pages = PyMuPDFLoader(PDF_PATH).load()
    for p in pages:
        p.page_content = limpar(p.page_content)
    pages = [p for p in pages if len(p.page_content) > 300]

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=200,
        add_start_index=True,
    )
    chunks = splitter.split_documents(pages)
    print(len(pages), "páginas |", len(chunks), "chunks")

    vectordb = Chroma.from_documents(chunks, embedding=embeddings_models, persist_directory=PERSIST_DIR)
else:
    vectordb = Chroma(persist_directory=PERSIST_DIR, embedding_function=embeddings_models)


AttributeError: 'Chroma' object has no attribute 'persist'

In [ ]:
retriever = vectordb.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 20},
)

chain = load_qa_chain(llm, chain_type="stuff")


In [ ]:
def ask(question, mostrar_fontes=True):
    docs = retriever.invoke(question)
    if mostrar_fontes:
        for d in docs:
            print(f"[pág. {d.metadata.get('page')}] {d.page_content[:200]!r}\n")
    result = chain.invoke({"input_documents": docs, "question": question})
    return result["output_text"]

In [ ]:
def ask_many(questions, mostrar_fontes=False):
    """Responde várias perguntas de uma vez, em paralelo."""
    docs_por_pergunta = retriever.batch(questions)
    resultados = chain.batch([
        {"input_documents": docs, "question": q}
        for q, docs in zip(questions, docs_por_pergunta)
    ])
    respostas = []
    for q, docs, r in zip(questions, docs_por_pergunta, resultados):
        fontes = sorted({d.metadata.get("page") for d in docs if d.metadata.get("page") is not None})
        respostas.append({"pergunta": q, "resposta": r["output_text"], "paginas": fontes})
        if mostrar_fontes:
            print(f"### {q}")
            for d in docs:
                print(f"[pág. {d.metadata.get('page')}] {d.page_content[:200]!r}\n")
    return respostas

In [ ]:
# Digite uma pergunta por linha e deixe a linha vazia para terminar.
# Também é possível colar várias de uma vez separadas por ";".
questions = []
while True:
    linha = input(f"Pergunta {len(questions) + 1} (vazio para terminar): ").strip()
    if not linha:
        break
    questions.extend(q.strip() for q in linha.split(";") if q.strip())

for i, r in enumerate(ask_many(questions), start=1):
    print(f"\n{i}. {r['pergunta']}")
    print(f"Resposta: {r['resposta']}")
    print(f"Páginas consultadas: {r['paginas']}")